限制工具调用次数，可以 `限制所有工具` 调用的总次数，也可以 `限制特定工具` 的调用次数。

作用如下：
- 避免过多调用某些昂贵的外部API
- 限制网络爬虫或数据库查询请求的数量
- 避免Agent陷入无限循环

退出行为有三种模式：
- error：直接抛异常
- end：结束整个会话
- continue：继续运行Agent，这是默认行为，此时Agent会将工具调用超出限制的信息传递给模型，后者自主决定后续行为，如果模型能力不足，可能导致死循环，为了避免这种情况，我们实现的fake server会以20%的概率输出正确响应，从而能终止循环。

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage,AIMessage,SystemMessage, ToolMessage
from langchain_deepseek import ChatDeepSeek
from langchain_core.runnables import RunnableConfig

from pydantic import BaseModel, Field, SecretStr
from typing import List, Union
from dotenv import load_dotenv
import os

load_dotenv(override=True)

model = ChatDeepSeek(
    model="any",
    base_url="http://127.0.0.1:8889",
    api_key=SecretStr(os.getenv('DEEPSEEK_API_KEY'))
)

class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")

class EventInfo(BaseModel):
    event_name: str = Field(description="事件名称")
    data: str = Field(description="事件发生日期")

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    tools=[],
    middleware=[
        ToolCallLimitMiddleware(
           # thread_limit=2, # 每个线程最多2次工具调用
            run_limit=2,  # 每次运行最多2次
            exit_behavior="end", # end - 优雅退出， error - 抛出异常, continue 继续运行Agent，这是默认行为
        )
    ],
    response_format=Union[ContactInfo, EventInfo],
)

def pretty_iterate_msg(messages: List[HumanMessage | AIMessage | SystemMessage | ToolMessage]):
    for msg in messages:
        msg.pretty_print()

config: RunnableConfig  = {
     "configurable": {
        "thread_id": "1"
    }
}

response = agent.invoke(
    {
        "messages": [HumanMessage("你好")],
    },
    config=config
)

pretty_iterate_msg(response["messages"])

================================ Human Message =================================

你好
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_1)
 Call ID: call_1
  Args:
    name: 小伙子
    email: xiaohuozi@qq.com
    phone: 12345678912
  EventInfo (call_2)
 Call ID: call_2
  Args:
    event_name: 问数项目启动会
    date: 2026-03-27
================================= Tool Message =================================
Name: ContactInfo

Error: Model incorrectly returned multiple structured responses (ContactInfo, EventInfo) when only one is expected.
 Please fix your mistakes.
================================= Tool Message =================================
Name: EventInfo

Error: Model incorrectly returned multiple structured responses (ContactInfo, EventInfo) when only one is expected.
 Please fix your mistakes.
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_1)
 Call ID: call_1
 